## Explorative Datenanalyse OpenligaDB

In diesem Notebook werden die strukturierten Daten von https://www.openligadb.de/ untersucht, um die Struktur und Inhalte der Daten so zu verstehen und auszuwählen, dass 

- sinnvolle Kennzahlen für das Statistik Tab definiert werden können (Minimalanforderungen erfüllen!)
- die Beispielfragen des Projekts dieser Quelle zugeordnet werden können um zu sehen, welche Informationen hier weitergegeben werden.
- eine Datei `prepare_openliga_data.py`entsteht, die eine Datei `openliga-tabellen.csv` erzeugt (zur Übergabe an Streamlit - Statistik Tab).


----------------------------------------

### Beispielfragen aus den Projektanforderungen
- siehe Functional Requirements (Chat) aus `NLP_and_GenAI_Case_2026.pdf`

Typische Chatanforderungen an OpenligaDB: 
- „Welche Spiele stehen am nächsten Bundesliga-Spieltag an?
- „Wie hat Bayern München in den letzten 5 Spielen abgeschnitten?“
- „Welche Teams stehen aktuell in der Bundesliga-Tabelle oben?“
- „Wann hat der 1. FC Köln sein letztes Heimspiel gewonnen?“

**WICHTIG für Chatfunktion**: 
- Alle oben genannten Fragen werden im Statistik Tab nicht beantwortet und sind auch nicht zugänglich. 
- Für diese Fragen ist ein erweiterter Datenbezug nötig!

--------------------------------------

### Setup

In [9]:
import requests
import pandas as pd
from IPython.display import display

### 1. Schritt: Datenbezug (Access to OpenLigaDB)

- 1. Bundesliga
- Saison 2023/24

Benennung der Saison bei OpenLigaDB: 

- URL-Parameter "2023"  =  Saison 2023/24
- URL-Parameter "2022"  =  Saison 2022/23
- URL-Parameter "2024"  =  Saison 2024/25

In [17]:
url = "https://api.openligadb.de/getbltable/bl1/2023"
response = requests.get(url)
data = response.json()
data

[{'teamInfoId': 6,
  'teamName': 'Bayer 04 Leverkusen',
  'shortName': 'Leverkusen',
  'teamIconUrl': 'https://www.bundesliga-reisefuehrer.de/sites/default/files/B04_Standard_Logo_RGB.png',
  'points': 90,
  'opponentGoals': 24,
  'goals': 89,
  'matches': 34,
  'won': 28,
  'lost': 0,
  'draw': 6,
  'goalDiff': 65},
 {'teamInfoId': 16,
  'teamName': 'VfB Stuttgart',
  'shortName': 'Stuttgart',
  'teamIconUrl': 'https://i.imgur.com/v0tkpNx.png',
  'points': 73,
  'opponentGoals': 39,
  'goals': 78,
  'matches': 34,
  'won': 23,
  'lost': 7,
  'draw': 4,
  'goalDiff': 39},
 {'teamInfoId': 40,
  'teamName': 'FC Bayern München',
  'shortName': 'Bayern',
  'teamIconUrl': 'https://upload.wikimedia.org/wikipedia/commons/1/1f/Logo_FC_Bayern_M%C3%BCnchen_%282002%E2%80%932017%29.svg',
  'points': 72,
  'opponentGoals': 45,
  'goals': 94,
  'matches': 34,
  'won': 23,
  'lost': 8,
  'draw': 3,
  'goalDiff': 49},
 {'teamInfoId': 1635,
  'teamName': 'RB Leipzig',
  'shortName': 'Leipzig',
  'teamI

- Spielplantabelle - nicht nützlich für ein erstes Dashboard: `url = "https://www.openligadb.de/api/getmatchdata/bl1/2023/1"`
- aber: `url = "https://api.openligadb.de/getbltable/bl1/2023"` (Tabellenstandtabelle)

```
Was du gerade hast:           Was du fürs Dashboard brauchst:
getmatchdata                  getbltable
─────────────────────         ─────────────────────
9 Zeilen = 9 einzelne         18 Zeilen = 18 Vereine
Spiele eines Spieltags        mit Saison-Endstand

Spalten: matchID, team1,      Spalten: teamName, points,
team2, goals, ...              goals, won, ...

In [18]:
df = pd.DataFrame(data)

display(df.head(20))


,teamInfoId,teamName,shortName,teamIconUrl,points,opponentGoals,goals,matches,won,lost,draw,goalDiff
0,6,Bayer 04 Leverkusen,Leverkusen,https://www.bundesliga-reisefuehrer.de/sites/d...,90,24,89,34,28,0,6,65
1,16,VfB Stuttgart,Stuttgart,https://i.imgur.com/v0tkpNx.png,73,39,78,34,23,7,4,39
2,40,FC Bayern München,Bayern,https://upload.wikimedia.org/wikipedia/commons...,72,45,94,34,23,8,3,49
3,1635,RB Leipzig,Leipzig,https://i.imgur.com/Rpwsjz1.png,65,39,77,34,19,7,8,38
4,7,Borussia Dortmund,Dortmund,https://upload.wikimedia.org/wikipedia/commons...,63,43,68,34,18,7,9,25
5,91,Eintracht Frankfurt,Frankfurt,https://i.imgur.com/X8NFkOb.png,47,50,51,34,11,9,14,1
6,175,TSG Hoffenheim,Hoffenheim,https://i.imgur.com/gF0PfEl.png,46,66,66,34,13,14,7,0
7,199,1. FC Heidenheim 1846,Heidenheim,https://upload.wikimedia.org/wikipedia/commons...,42,55,50,34,10,12,12,-5
8,134,SV Werder Bremen,Bremen,https://upload.wikimedia.org/wikipedia/commons...,42,54,48,34,11,14,9,-6
9,112,SC Freiburg,Freiburg,https://i.imgur.com/r3mvi0h.png,42,58,45,34,11,14,9,-13


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   teamInfoId     18 non-null     int64
 1   teamName       18 non-null     str  
 2   shortName      18 non-null     str  
 3   teamIconUrl    18 non-null     str  
 4   points         18 non-null     int64
 5   opponentGoals  18 non-null     int64
 6   goals          18 non-null     int64
 7   matches        18 non-null     int64
 8   won            18 non-null     int64
 9   lost           18 non-null     int64
 10  draw           18 non-null     int64
 11  goalDiff       18 non-null     int64
dtypes: int64(9), str(3)
memory usage: 3.3 KB


-----------------------------------------------

### Data Dictionary

| Spalte         | Datentyp | Beschreibung                                |
|----------------|----------|----------------------------------------------|
| teamInfoId     | int64    | Eindeutige ID des Vereins                    |
| teamName       | str      | Vollständiger Vereinsname                    |
| shortName      | str      | Kurzname des Vereins                         |
| teamIconUrl    | str      | URL zum Vereinslogo                          |
| points         | int64    | Punkte in der Saison                         |
| opponentGoals  | int64    | Gegentore (Tore die der Verein bekommen hat) |
| goals          | int64    | Tore die der Verein geschossen hat           |
| matches        | int64    | Anzahl gespielter Spiele                     |
| won            | int64    | Anzahl Siege                                 |
| lost           | int64    | Anzahl Niederlagen                           |
| draw           | int64    | Anzahl Unentschieden                         |
| goalDiff       | int64    | Tordifferenz (goals - opponentGoals)         |

----------------------------------------

### Überlegungen fürs Dashboard: 

```
┌─────────────────────────────────────┐
│  Filter: [Saison ▾]  [Verein ▾]      │
└─────────────────────────────────────┘

┌──────────┬──────────┬──────────┐
│ Platz    │ Punkte   │ Siege    │
│   3      │   68     │   21     │
└──────────┴──────────┴──────────┘

┌──────────┬──────────┐
│ Tore     │ Gegentore│
│   89     │   32     │
└──────────┴──────────┘

[Barchart: Tore vs. Gegentore über alle Vereine]

---------------------------------

### Datenauswahl fürs Dashboard

BEHALTEN für KPIs:
- teamName        → für den Filter "Verein"
- points          → KPI "Punkte"
- goals           → KPI "Tore"
- opponentGoals   → KPI "Gegentore"
- won             → KPI "Siege"

In [ ]:
df.drop(columns=['teamInfoId', 'shortName', 'teamIconUrl', 'matches', 'lost', 'draw', 'goalDiff'], inplace=True)

In [22]:
df.head(20)

,teamName,points,opponentGoals,goals,won
0,Bayer 04 Leverkusen,90,24,89,28
1,VfB Stuttgart,73,39,78,23
2,FC Bayern München,72,45,94,23
3,RB Leipzig,65,39,77,19
4,Borussia Dortmund,63,43,68,18
5,Eintracht Frankfurt,47,50,51,11
6,TSG Hoffenheim,46,66,66,13
7,1. FC Heidenheim 1846,42,55,50,10
8,SV Werder Bremen,42,54,48,11
9,SC Freiburg,42,58,45,11


Eine Spalte finale Platzierung soll noch berechnet und hinzugefügt werden. Dafür müssen die Punkte der Teams sortiert und die Platzierung zugewiesen werden. Punktegleichstand wird dann 

------------------------------------------

### Mehrere Saisonen laden für Filter

In [30]:
def hole_tabelle(start_jahr: int) -> pd.DataFrame:
    url = f"https://api.openligadb.de/getbltable/bl1/{start_jahr}"
    daten = requests.get(url).json()
    df = pd.DataFrame(daten)
    # Lesbares Label für den Filter erzeugen
    df["saison_label"] = f"{start_jahr}/{str(start_jahr+1)[-2:]}"
    return df

df_2024 = hole_tabelle(2024)  # Saison 2024/25
df_2023 = hole_tabelle(2023)  # Saison 2023/24
df_2022 = hole_tabelle(2022)  # Saison 2022/23

# Alle zusammenführen
df_gesamt = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)


In [31]:
df_gesamt.head()

,teamInfoId,teamName,shortName,teamIconUrl,points,opponentGoals,goals,matches,won,lost,draw,goalDiff,saison_label
0,40,FC Bayern München,Bayern,https://upload.wikimedia.org/wikipedia/commons...,71,38,92,34,21,5,8,54,2022/23
1,7,Borussia Dortmund,Dortmund,https://upload.wikimedia.org/wikipedia/commons...,71,44,83,34,22,7,5,39,2022/23
2,1635,RB Leipzig,Leipzig,https://i.imgur.com/Rpwsjz1.png,66,41,64,34,20,8,6,23,2022/23
3,80,1. FC Union Berlin,Union Berlin,https://assets.dfb.de/uploads/000/018/232/smal...,62,38,51,34,18,8,8,13,2022/23
4,112,SC Freiburg,Freiburg,https://i.imgur.com/r3mvi0h.png,59,44,51,34,17,9,8,7,2022/23


---------------------------------------------

### Pipeline bauen (prepare_openliga_data.py)

- 1. Schritt: Mehrere Saisonen laden und zusammenführen
- 2. Schritt: unnötige Spalten löschen
- 3. Schritt: Finale Platzierung berechnen und Spalte ergänzen
- 4. Schritt: abspeichern als csv für Übergabe an Streamlit (Statistik-Tab)

In [ ]:
"""
prepare_openliga_data.py

Lädt mehrere Bundesliga-Saisons von OpenLigaDB, bereinigt die Daten,
berechnet die finale Platzierung und speichert das Ergebnis als CSV
für den Statistik-Tab in Streamlit.

Ausführen:
    uv run python prepare_openliga_data.py
"""

import requests
import pandas as pd


# ─────────────────────────────────────────────────────────────
# Schritt 1: Eine Saison laden
# ─────────────────────────────────────────────────────────────
def hole_tabelle(start_jahr: int) -> pd.DataFrame:
    """Lädt die Tabelle einer Bundesliga-Saison von OpenLigaDB."""
    url = f"https://api.openligadb.de/getbltable/bl1/{start_jahr}"
    daten = requests.get(url).json()
    df = pd.DataFrame(daten)
    # Lesbares Label für den Filter erzeugen, z.B. "2023/24"
    df["saison_label"] = f"{start_jahr}/{str(start_jahr + 1)[-2:]}"
    return df


# ─────────────────────────────────────────────────────────────
# Schritt 2: Unnötige Spalten entfernen
# ─────────────────────────────────────────────────────────────
def bereinige_spalten(df: pd.DataFrame) -> pd.DataFrame:
    """Behält nur die für das Dashboard relevanten Spalten."""
    relevante_spalten = [
        "teamName",
        "saison_label",
        "points",
        "goals",
        "opponentGoals",
        "goalDiff",
        "won",
        "lost",
        "draw",
    ]
    return df[relevante_spalten]


# ─────────────────────────────────────────────────────────────
# Schritt 3: Finale Platzierung berechnen
# ─────────────────────────────────────────────────────────────
def berechne_platzierung(df: pd.DataFrame) -> pd.DataFrame:
    """
    Berechnet die Tabellenplatzierung pro Saison.
    Sortierregel: erst Punkte, bei Gleichstand Tordifferenz.
    """
    df_mit_platz = []
    for saison in df["saison_label"].unique():
        df_saison = df[df["saison_label"] == saison].copy()
        df_saison = df_saison.sort_values(
            ["points", "goalDiff"], ascending=[False, False]
        ).reset_index(drop=True)
        df_saison["platzierung"] = df_saison.index + 1
        df_mit_platz.append(df_saison)
    return pd.concat(df_mit_platz, ignore_index=True)


# ─────────────────────────────────────────────────────────────
# Hauptablauf
# ─────────────────────────────────────────────────────────────
def main():
    # Schritt 1: Mehrere Saisons laden und zusammenführen
    print("Lade Saisons von OpenLigaDB...")
    saisons = [2022, 2023, 2024]
    df_liste = [hole_tabelle(jahr) for jahr in saisons]
    df_gesamt = pd.concat(df_liste, ignore_index=True)
    print(f"  ✓ {len(df_gesamt)} Zeilen aus {len(saisons)} Saisons geladen")

    # Schritt 2: Unnötige Spalten löschen
    df_gesamt = bereinige_spalten(df_gesamt)
    print(f"  ✓ Spalten bereinigt: {list(df_gesamt.columns)}")

    # Schritt 3: Finale Platzierung berechnen
    df_gesamt = berechne_platzierung(df_gesamt)
    print("  ✓ Platzierung berechnet")

    # Schritt 4: Als CSV speichern
    ausgabe_pfad = "openliga_tabellen.csv"
    df_gesamt.to_csv(ausgabe_pfad, index=False)
    print(f"  ✓ Gespeichert unter: {ausgabe_pfad}")

    print("\nVorschau:")
    print(df_gesamt.head(10).to_string(index=False))


if __name__ == "__main__":
    main()